In [36]:
from deepagents import create_deep_agent
from deepagents.backends.filesystem import FilesystemBackend
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

# Load environment variables from a .env file
load_dotenv()

# 1. Initialize your language model
llm = ChatOpenAI(model="gpt-5-mini-2025-08-07")


In [37]:
from pathlib import Path
BASE_DIR = Path.cwd()
print(BASE_DIR)

/Users/aritrasen/Documents/code/github/agent_harness


In [42]:
# 2. Give the agent a pluggable filesystem backend so it can read/write files
# This grants the agent tools like read_file, write_file, and ls.
backend = FilesystemBackend(root_dir=str(BASE_DIR))

# 3. Create the Deep Agent harness
agent = create_deep_agent(
    model=llm,
    system_prompt="You are a helpful and autonomous task orchestrator.",
    backend=backend,
    
    # Loads project-level conventions into the system prompt
    memory=str(BASE_DIR / "AGENTS.md"),
    
    # Scans the folder, parses YAML frontmatter, and makes skills available to the agent
    skills=str(BASE_DIR / "skills") 
)

/var/folders/57/y3hsdcy135b1wr5gzcw3b3kr0000gn/T/ipykernel_40395/3691406089.py:3: DeprecationWarning: FilesystemBackend virtual_mode default will change in deepagents 0.5.0; please specify virtual_mode explicitly. Note: virtual_mode is for virtual path semantics (e.g., CompositeBackend routing) and optional path-based guardrails; it does not provide sandboxing or process isolation. Security note: leaving virtual_mode=False allows absolute paths and '..' to bypass root_dir. Consult the API reference for details.
  backend = FilesystemBackend(root_dir=str(BASE_DIR))


In [43]:
# 4. Invoke the agent
if __name__ == "__main__":
    response = agent.invoke({
        "messages": [
            {"role": "user", "content": "I have a long article in " + str(BASE_DIR / "workspace" / "article.txt") + ". Please summarize it for me."}
        ]
    })
    
print("\n--- Agent Execution Complete ---")

OSError: [Errno 22] Invalid argument: '/.resolve/SKILL.md'

In [31]:
print(response['messages'][-1])

content="I couldn't find /workspace/article.txt — the workspace directory is empty. Please either:\n\n- Upload or paste the article text here, or\n- Give the correct absolute path to the file in the workspace.\n\nIf the article is very long, say whether you want a short (1–3 sentences), medium (1 paragraph), or detailed (multi-paragraph) summary." additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 211, 'prompt_tokens': 6814, 'total_tokens': 7025, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 6784}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZbCKMwr4yRSZqpx7T148N7wi4bNc', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dd3ea-47f4-79d2-b0b3-0ad3a39ca2e0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_t

In [32]:
response

{'messages': [HumanMessage(content='I have a long article in workspace/article.txt. Please summarize it for me.', additional_kwargs={}, response_metadata={}, id='5dafc06a-7433-463d-adda-5a6ae5f8ab83'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_G98UoX8RniwODj2FMQswVgDZ', 'function': {'arguments': '{"path":"/workspace"}', 'name': 'ls'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 6785, 'total_tokens': 6809, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 6656}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DZbCIjeW9ZQhFeeo2dTEtaGKx24QQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dd3ea-3f8a-7a03-9ae6-a41d137ef6aa-0', tool_calls=[{'name': 'ls', 'args': 

In [35]:
with open(str(BASE_DIR / "AGENTS.md"), "r") as f:
    print(f.read())

# Project Context: Content Processing Pipeline

## Conventions
- You are a generic Deep Agent orchestrator. 
- Use the `write_todos` tool to plan out complex requests before acting.
- All intermediate and final output files MUST be written to the `./workspace/` directory.
- Never execute external shell commands without asking for permission first.

## Capabilities
- You have access to specialized skills in the `skills/` directory. 
- You only see the names and descriptions initially. For specialized tasks, load the matching `SKILL.md` first.
